In [16]:
# Cell 2: Upload your dataset file from your local machine
from google.colab import files

print("Upload your qwen_actionable_train.jsonl file")
uploaded = files.upload()

# Verify it loaded correctly
import json

samples = []
with open("qwen_actionable_train.jsonl", "r") as f:
    for line in f:
        samples.append(json.loads(line.strip()))

print(f"✅ Loaded {len(samples)} training examples")
print("\nSample entry:")
print(samples[0]["text"][:200])

Upload your qwen_actionable_train.jsonl file


Saving qwen_actionable_train.jsonl to qwen_actionable_train (1).jsonl
✅ Loaded 1000 training examples

Sample entry:
<|im_start|>user
error entering a goddamn different shipping address. Change delivery destination to 742 Evergreen Terrace, order #764283<|im_end|>
<|im_start|>assistant
{"intent": "set_up_shipping_ad


In [17]:
# Cell 3: Check GPU and import everything
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [18]:
# Cell 4: Load and inspect dataset
dataset = load_dataset("json", data_files="qwen_actionable_train.jsonl", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print("\n--- Example 1 ---")
print(dataset[0]["text"])
print("\n--- Example 2 ---")
print(dataset[1]["text"])

# Split into train/validation (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"\nTrain: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Dataset size: 1000 examples
Columns: ['text']

--- Example 1 ---
<|im_start|>user
error entering a goddamn different shipping address. Change delivery destination to 742 Evergreen Terrace, order #764283<|im_end|>
<|im_start|>assistant
{"intent": "set_up_shipping_address", "order_id": "764283", "new_address": "742 Evergreen Terrace"}<|im_end|>

--- Example 2 ---
<|im_start|>user
I  have to edit order {{Order Number}} (Order ID: #243565)<|im_end|>
<|im_start|>assistant
{"intent": "change_order", "order_id": "243565"}<|im_end|>

Train: 900 | Eval: 100


In [19]:
# Cell 5: Load Qwen2.5-1.5B-Instruct in 4-bit (fits on free Colab T4 with ~6GB VRAM)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization config — essential for fitting in Colab free tier
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # saves extra VRAM
    bnb_4bit_quant_type="nf4",        # NormalFloat4 is best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # important for causal LM training

# Load model
print("Loading model (this takes ~2 minutes on Colab)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training (adds gradient checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded")
print(f"Model dtype: {model.dtype}")

Loading model (this takes ~2 minutes on Colab)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Model loaded
Model dtype: torch.float32


In [20]:
# Cell 6: Configure and attach LoRA adapter
# We only train ~1-2% of total parameters — the rest stay frozen

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # rank — higher = more capacity, more VRAM
    lora_alpha=32,                 # scaling factor (usually 2x r)
    lora_dropout=0.05,
    bias="none",
    # Target the attention projection layers — standard for Qwen
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# Show how many params are actually being trained
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print(f"Trainable %      : {100 * trainable / total:.2f}%")

Trainable params : 18,464,768
Total params     : 1,562,179,072
Trainable %      : 1.18%


In [31]:
# Cell 7: Fixed for newer TRL versions

OUTPUT_DIR = "./qwen-ecommerce-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # --- Core training ---
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,

    # --- Optimizer ---
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,               # ← replaced warmup_ratio (avoids deprecation warning)
    lr_scheduler_type="cosine",

    # --- Precision ---
    bf16=True,
    fp16=False,

    # --- Logging + eval ---
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- Data ---
    dataset_text_field="text",
    max_length=512,            # ← stays here in SFTConfig, NOT in SFTTrainer
    packing=False,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # ← no max_seq_length here, no tokenizer kwarg in newer TRL
    processing_class=tokenizer,    # ← replaces 'tokenizer=' in TRL ≥ 0.12
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

/tmp/ipykernel_3996/3052233493.py:5: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


🚀 Starting training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.663482,0.648556,0.660015,0.810505,39161.000000
100,0.552989,0.558469,0.570702,0.818596,77691.000000
150,0.473493,0.542815,0.503460,0.826002,116322.000000
171,0.480759,0.542640,0.504091,0.825216,131997.000000


✅ Training complete!


In [32]:
# Cell 8: Save the LoRA adapter weights (NOT the full model — adapters are tiny, ~20MB)
ADAPTER_SAVE_PATH = "./qwen-ecommerce-lora-final"

trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ Adapter saved to {ADAPTER_SAVE_PATH}")

# Check what was saved
import os
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(f"{ADAPTER_SAVE_PATH}/{f}") / 1e6
    print(f"  {f:40s} {size:.2f} MB")

✅ Adapter saved to ./qwen-ecommerce-lora-final
  chat_template.jinja                      0.00 MB
  README.md                                0.01 MB
  tokenizer_config.json                    0.00 MB
  adapter_config.json                      0.00 MB
  adapter_model.safetensors                73.91 MB
  tokenizer.json                           11.42 MB


In [33]:
# Cell 9: Test the fine-tuned model with some examples

from peft import PeftModel

# Reload base model + adapter for clean inference test
print("Loading model for inference test...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
inf_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
inf_model.eval()

def predict(user_message: str) -> str:
    prompt = (
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(inf_model.device)

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,        # greedy — deterministic JSON output
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        )

    # Decode only the newly generated tokens (not the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Validate the output is parseable JSON
    try:
        parsed = json.loads(raw)
        return json.dumps(parsed, indent=2)
    except json.JSONDecodeError:
        return f"[Warning: non-JSON output]\n{raw}"

# --- Run test cases ---
test_cases = [
    "I want to cancel my order #789456",
    "Please change my delivery address to 10 Baker Street for order #334455",
    "Delete my account, my email is john@example.com",
    "I need to modify order #998877",
]

print("=" * 55)
for msg in test_cases:
    print(f"\n USER: {msg}")
    print(f" BOT : {predict(msg)}")
    print("-" * 55)

Loading model for inference test...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



 USER: I want to cancel my order #789456


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


 BOT : {
  "intent": "cancel_order",
  "order_id": "789456"
}
-------------------------------------------------------

 USER: Please change my delivery address to 10 Baker Street for order #334455
 BOT : {
  "intent": "change_shipping_address",
  "order_id": "334455",
  "new_address": "10 Baker Street"
}
-------------------------------------------------------

 USER: Delete my account, my email is john@example.com
 BOT : {
  "intent": "delete_account",
  "email": "john@example.com"
}
-------------------------------------------------------

 USER: I need to modify order #998877
 BOT : {
  "intent": "change_order",
  "order_id": "998877"
}
-------------------------------------------------------


In [35]:
# Cell 11 (optional): Push adapter to HF Hub for deployment
# You'll need your HF write token from huggingface.co/settings/tokens

from huggingface_hub import login

login()  # prompts for your HF token

inf_model.push_to_hub("mahdi2020/qwen2.5-1.5B-ecommerce-lora-fine-tuned")
tokenizer.push_to_hub("mahdi2020/qwen2.5-1.5B-ecommerce-lora-fine-tuned")

print("✅ Pushed to HF Hub!")
print("Load it later with: PeftModel.from_pretrained(base_model, 'mahdi2020/qwen-ecommerce-lora')")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpjosvos6t/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

✅ Pushed to HF Hub!
Load it later with: PeftModel.from_pretrained(base_model, 'mahdi2020/qwen-ecommerce-lora')


In [36]:
# Cell 10: Zip and download the adapter so you have it locally
import shutil

shutil.make_archive("qwen_lora_adapter", "zip", ADAPTER_SAVE_PATH)

from google.colab import files
files.download("qwen_lora_adapter.zip")
print("✅ Download started")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started
